# Notebook 07 — MURA XR\_HAND Control Dataset Preparation

**Project:** P63 – Multimodal Deep Learning for Autoimmune Disease Diagnosis  
**Phase:** RA — adding MURA negative XR\_HAND images as control/non-RA class  

---

## Purpose

The existing RAM-H1200-v1 RA dataset contains only 80 Non-RA images versus 1120 RA images (~14:1 imbalance).  
This notebook prepares **MURA XR\_HAND negative** (normal) radiographs as an additional control class,
to enable a fully balanced 1:1 RA vs Control experiment.

## What is and is NOT changed

| Item | Changed? |
|---|---|
| Original RA dataset (`Dataset/ra/`) | ❌ NOT touched |
| Existing train/val/test RA manifests | ❌ NOT touched |
| Existing notebooks (01–06) | ❌ NOT touched |
| Existing `src/` modules | ❌ NOT touched |
| Original MURA files | ❌ NOT touched |
| `Dataset/mura_controls/` | ✅ NEW — created by this notebook |
| `outputs/reports/mura_controls_manifest.csv` | ✅ NEW — created by this notebook |

## MURA label clarification

> **MURA Negative** = the study was labelled **normal** (no musculoskeletal abnormality detected).  
> This does NOT clinically confirm the absence of Rheumatoid Arthritis.  
> These images are used solely as **normal/control radiograph examples** for this classification experiment.

## Split strategy (Option C — patient-level, zero leakage)

| Target split | MURA source | Images | Patients |
|---|---|---|---|
| val controls | MURA `valid/XR_HAND` negatives | 136 | 49 |
| test controls | Held-out `train/XR_HAND` patients | 265 | 97 |
| train controls | Remaining `train/XR_HAND` patients | 719 | 254 |

No patient appears in more than one split.  
Random seed = 42 throughout.

---

**Prerequisites:** MURA dataset must be present at `MURA_ MSK Xrays_files/`.  
Run after Notebooks 01–04 (RA manifests must exist in `outputs/reports/`).

---
## Section 1 — Project Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)

---
## Section 2 — Imports

In [ ]:
import os
import random
import shutil
import hashlib
import warnings
from collections import OrderedDict, Counter
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style='whitegrid', palette='muted')
Image.MAX_IMAGE_PIXELS = None

# ── Paths ─────────────────────────────────────────────────────────────────────
MURA_ROOT    = PROJECT_ROOT / 'MURA_ MSK Xrays_files'
DEST_ROOT    = PROJECT_ROOT / 'Dataset' / 'mura_controls'
REPORTS_DIR  = PROJECT_ROOT / 'outputs' / 'reports'
PLOTS_DIR    = PROJECT_ROOT / 'outputs' / 'plots'

MANIFEST_PATH = REPORTS_DIR / 'mura_controls_manifest.csv'

# Existing RA manifests (NOT modified)
RA_TRAIN_CSV  = REPORTS_DIR / 'train_manifest.csv'
RA_VAL_CSV    = REPORTS_DIR / 'val_manifest.csv'
RA_TEST_CSV   = REPORTS_DIR / 'test_manifest.csv'

RANDOM_SEED   = 42
IMG_EXTS      = {'.png', '.jpg', '.jpeg', '.bmp'}
TARGETS       = {'train': 719, 'val': 136, 'test': 265}

random.seed(RANDOM_SEED)

# Verify all required paths exist
for p in [MURA_ROOT, RA_TRAIN_CSV, RA_VAL_CSV, RA_TEST_CSV]:
    assert p.exists(), f'Required path not found: {p}'

print('All paths verified.')
print(f'MURA root     : {MURA_ROOT}')
print(f'Control dest  : {DEST_ROOT}')
print(f'Manifest out  : {MANIFEST_PATH}')

---
## Section 3 — Dataset Inspection

### 3a — Existing RA split (do not modify)

In [ ]:
ra_train = pd.read_csv(RA_TRAIN_CSV)
ra_val   = pd.read_csv(RA_VAL_CSV)
ra_test  = pd.read_csv(RA_TEST_CSV)

print('=== EXISTING RA DATASET (unchanged) ===')
print(f'{"Split":<8} {"Total":>8} {"RA":>8} {"Non-RA":>8} {"RA%":>7}')
print('-' * 45)
for name, df in [('train', ra_train), ('val', ra_val), ('test', ra_test)]:
    tot = len(df)
    ra  = int((df['isRA'] == 1).sum())
    nra = int((df['isRA'] == 0).sum())
    print(f'{name:<8} {tot:>8} {ra:>8} {nra:>8} {ra/tot*100:>6.1f}%')
print('-' * 45)
total_ra = int((ra_train['isRA']==1).sum() + (ra_val['isRA']==1).sum() + (ra_test['isRA']==1).sum())
print(f'Total RA images: {total_ra}')
print()
print('NOTE: These manifests are NOT modified by this notebook.')

### 3b — MURA dataset structure inspection

In [ ]:
print('=== MURA DIRECTORY STRUCTURE ===')
for split in ['train', 'valid']:
    split_dir = MURA_ROOT / split
    if not split_dir.exists():
        print(f'  {split}: NOT FOUND')
        continue
    body_parts = [d.name for d in split_dir.iterdir() if d.is_dir()]
    print(f'  {split}/ → body parts: {body_parts}')
print()
print('Only XR_HAND is present — no filtering by body region required.')
print('XR_HAND (hand X-rays) is directly compatible with the RAM-H1200 RA hand dataset.')

In [ ]:
# ── Count all MURA XR_HAND studies and images ─────────────────────────────────
def count_mura_split(split_name):
    base = MURA_ROOT / split_name / 'XR_HAND'
    pos_studies = neg_studies = 0
    pos_images  = neg_images  = 0
    pos_patients = set()
    neg_patients = set()
    multi_study  = []

    for pt in sorted(base.iterdir()):
        if not pt.is_dir():
            continue
        studies = [s for s in pt.iterdir() if s.is_dir()]
        if len(studies) > 1:
            multi_study.append((pt.name, [s.name for s in studies]))
        for study in studies:
            imgs = [f for f in study.iterdir() if f.suffix.lower() in IMG_EXTS]
            if 'positive' in study.name.lower():
                pos_studies += 1
                pos_images  += len(imgs)
                pos_patients.add(pt.name)
            elif 'negative' in study.name.lower():
                neg_studies += 1
                neg_images  += len(imgs)
                neg_patients.add(pt.name)
    return {
        'split'       : split_name,
        'pos_studies' : pos_studies,  'pos_images' : pos_images,  'pos_patients': len(pos_patients),
        'neg_studies' : neg_studies,  'neg_images' : neg_images,  'neg_patients': len(neg_patients),
        'multi_study' : multi_study,
    }

mura_stats = {s: count_mura_split(s) for s in ['train', 'valid']}

print('=== MURA XR_HAND FULL INVENTORY ===')
print(f'{"Split":<8} {"Neg_pts":>8} {"Neg_imgs":>10} {"Pos_pts":>8} {"Pos_imgs":>10} {"Multi_study":>12}')
print('-' * 65)
for sp, st in mura_stats.items():
    print(f'{sp:<8} {st["neg_patients"]:>8} {st["neg_images"]:>10} '
          f'{st["pos_patients"]:>8} {st["pos_images"]:>10} {len(st["multi_study"]):>12}')
print()
print('Image format: PNG | Mode: RGB (variable sizes)')
print('Label encoding: study folder name contains "positive" or "negative"')
print()
print('Multi-study patient examples (train):')
for p, s in mura_stats['train']['multi_study'][:3]:
    print(f'  {p} -> {s}')

---
## Section 4 — Metadata Inspection

In [ ]:
# ── Sample image properties ───────────────────────────────────────────────────
sample_imgs = list((MURA_ROOT / 'train' / 'XR_HAND').rglob('*.png'))[:5]

print('=== SAMPLE MURA IMAGE PROPERTIES ===')
print(f'{"Path":<55} {"Mode":<6} {"Size"}')
print('-' * 80)
for p in sample_imgs:
    with Image.open(p) as img:
        mode = img.mode
        size = f'{img.width}x{img.height}'
    label = 'negative' if 'negative' in p.parent.name else 'positive'
    print(f'{str(p)[-55:]:<55} {mode:<6} {size}  [{label}]')
print()
print('MURA images: PNG format, RGB mode, variable dimensions')
print('RA images  : BMP format, Grayscale (L) mode, variable dimensions')
print()
print('Preprocessing needed for MURA:')
print('  - Convert RGB -> L (grayscale) for consistency')
print('  - Resize to 224x224 at training time (handled by DataLoader, not here)')
print('  - Normalisation applied at training time via existing normalisation stats')

---
## Section 5 — Patient-Level Analysis

In [ ]:
# ── Build negative patient maps ───────────────────────────────────────────────
def build_neg_patient_map(mura_split):
    base = MURA_ROOT / mura_split / 'XR_HAND'
    pmap = OrderedDict()
    for pt in sorted(base.iterdir()):
        if not pt.is_dir():
            continue
        studies = []
        for study in sorted(pt.iterdir()):
            if not study.is_dir():
                continue
            if 'negative' not in study.name.lower():
                continue
            imgs = sorted([f for f in study.iterdir()
                           if f.suffix.lower() in IMG_EXTS])
            if imgs:
                studies.append({'study_id': study.name,
                                 'study_path': study,
                                 'images': imgs})
        if studies:
            pmap[pt.name] = studies
    return pmap


mura_valid_map = build_neg_patient_map('valid')
mura_train_map = build_neg_patient_map('train')

# Image counts
def count_imgs(pmap):
    return sum(len(s['images']) for studies in pmap.values() for s in studies)

print('=== MURA NEGATIVE PATIENT MAP ===')
print(f'  MURA valid  : {len(mura_valid_map):4d} patients | {count_imgs(mura_valid_map):5d} images')
print(f'  MURA train  : {len(mura_train_map):4d} patients | {count_imgs(mura_train_map):5d} images')
print()

# Images per patient distribution
def imgs_per_patient(pmap):
    return [sum(len(s['images']) for s in studies) for studies in pmap.values()]

for name, pmap in [('MURA valid', mura_valid_map), ('MURA train', mura_train_map)]:
    counts = imgs_per_patient(pmap)
    arr    = np.array(counts)
    print(f'{name} — images per patient: '
          f'min={arr.min()}  median={int(np.median(arr))}  '
          f'mean={arr.mean():.1f}  max={arr.max()}')

In [ ]:
# Patient overlap between MURA valid and MURA train
overlap_vt = set(mura_valid_map.keys()) & set(mura_train_map.keys())
print(f'Patient overlap (MURA valid ∩ MURA train): {len(overlap_vt)}')
if overlap_vt:
    print('WARNING: overlapping patients found — must be handled during split.')
else:
    print('PASS: MURA valid and train patient sets are already disjoint.')

---
## Section 6 — Body Region Selection

**Why XR\_HAND only?**

The RAM-H1200-v1 RA dataset consists exclusively of **hand radiographs**.
Rheumatoid Arthritis primarily affects small joints of the hands and wrists, and standard radiographic
assessment of RA uses hand/wrist X-rays to evaluate joint-space narrowing and erosions.

The locally downloaded MURA dataset contains **only `XR_HAND`** — no other body parts were downloaded.
This removes the need for any filtering decision: all available MURA images are anatomically appropriate.

| Body region | Relevance to RA hand X-ray classification | Selected? |
|---|---|---|
| XR_HAND | Direct anatomical match — hand radiographs | ✅ YES |
| XR_WRIST | Adjacent region, also affected in RA | Not present locally |
| XR_SHOULDER, XR_ELBOW, XR_FOREARM, XR_HUMERUS, XR_FINGER | Less directly relevant | Not present locally |

---
## Section 7 — Positive/Negative Separation

In [ ]:
# Visualise positive vs negative counts
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('MURA XR_HAND — Positive vs Negative Studies', fontsize=13, fontweight='bold')

for ax, (split_label, stats) in zip(axes, mura_stats.items()):
    labels  = ['Negative\n(Normal)', 'Positive\n(Abnormal)']
    counts  = [stats['neg_images'], stats['pos_images']]
    colours = ['#4C72B0', '#C44E52']
    bars    = ax.bar(labels, counts, color=colours, edgecolor='white', width=0.5)
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 5,
                str(int(b.get_height())), ha='center', va='bottom', fontsize=11)
    ax.set_title(f'MURA {split_label}/')
    ax.set_ylabel('Images')
    ax.set_ylim(0, max(counts) * 1.18)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_mura_pos_neg_split.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: 07_mura_pos_neg_split.png')
print()
print('We use ONLY the Negative (normal) images as controls.')
print('Positive (abnormal) images are completely excluded.')

---
## Section 8 — Patient-Level Split Selection (Seed = 42)

In [ ]:
def select_patients(patient_map, target_images, exclude_patients=None):
    """
    Greedy patient-level selection.
    Shuffles patients (seed already set globally), adds whole patients until
    enough images are accumulated, then trims the image list to exact target.
    Returns (selected_patient_ids, records_list).
    Each record: {patient_id, study_id, study_path, image_path}
    """
    if exclude_patients is None:
        exclude_patients = set()

    candidates = [p for p in patient_map if p not in exclude_patients]
    random.shuffle(candidates)

    selected_pts = []
    records      = []

    for pid in candidates:
        if len(records) >= target_images:
            break
        for study in patient_map[pid]:
            for img_path in study['images']:
                records.append({
                    'patient_id': pid,
                    'study_id'  : study['study_id'],
                    'study_path': study['study_path'],
                    'image_path': img_path,
                })
        selected_pts.append(pid)

    records      = records[:target_images]
    selected_pts = list(dict.fromkeys(r['patient_id'] for r in records))
    return selected_pts, records


random.seed(RANDOM_SEED)   # reset seed before selection

# VAL — from MURA valid
val_pts, val_records = select_patients(mura_valid_map, TARGETS['val'])

# TEST — held-out from MURA train
random.seed(RANDOM_SEED)   # same seed guarantees same order as in preparation script
test_pts, test_records = select_patients(mura_train_map, TARGETS['test'])
test_pt_set = set(test_pts)

# TRAIN — remaining MURA train patients
train_pts, train_records = select_patients(
    mura_train_map, TARGETS['train'], exclude_patients=test_pt_set
)

print('=== PATIENT SELECTION RESULTS ===')
print(f'{"Pool":<8} {"Images":>8} {"Patients":>10} {"Target":>8} {"Match?":>8}')
print('-' * 50)
for name, records, pts, tgt in [
        ('val',   val_records,   val_pts,   TARGETS['val']),
        ('test',  test_records,  test_pts,  TARGETS['test']),
        ('train', train_records, train_pts, TARGETS['train']),
]:
    ok = len(records) == tgt
    print(f'{name:<8} {len(records):>8} {len(pts):>10} {tgt:>8} {"YES" if ok else "NO":>8}')
print()

assert len(val_records)   == TARGETS['val'],   'Val target not met'
assert len(test_records)  == TARGETS['test'],  'Test target not met'
assert len(train_records) == TARGETS['train'], 'Train target not met'
print('All targets met.')

---
## Section 9 — Leakage Verification

In [ ]:
val_pt_set   = set(val_pts)
test_pt_set  = set(test_pts)
train_pt_set = set(train_pts)

vt   = val_pt_set  & test_pt_set
vtr  = val_pt_set  & train_pt_set
ttr  = test_pt_set & train_pt_set

print('=== PATIENT LEAKAGE VERIFICATION ===')
print(f'  Val   ∩ Test  : {len(vt):4d}  (must be 0)')
print(f'  Val   ∩ Train : {len(vtr):4d}  (must be 0)')
print(f'  Test  ∩ Train : {len(ttr):4d}  (must be 0)')
print()

if vt or vtr or ttr:
    raise RuntimeError('PATIENT LEAKAGE DETECTED — do not proceed.')
print('PASS: Zero patient leakage across all splits.')

---
## Section 10 — Image Validation

Each selected image is opened with PIL to detect corrupted files before copying.

In [ ]:
print('Validating all selected images with PIL...')
print('(This reads every image header — may take 1–2 minutes for 1120 images)')

all_records = []
for split_name, records in [('val', val_records), ('test', test_records), ('train', train_records)]:
    for rec in records:
        rec['split'] = split_name
        all_records.append(rec)

valid_recs    = []
corrupted_recs= []
modes_seen    = set()

for i, rec in enumerate(all_records):
    src = rec['image_path']
    try:
        with Image.open(src) as img:
            img.verify()
        with Image.open(src) as img:
            rec['img_mode'] = img.mode
            rec['img_size'] = f'{img.width}x{img.height}'
            modes_seen.add(img.mode)
        rec['preprocessing_status'] = 'ok'
        valid_recs.append(rec)
    except Exception as e:
        rec['preprocessing_status'] = f'corrupted:{e}'
        rec['img_mode'] = ''
        rec['img_size'] = ''
        corrupted_recs.append(rec)
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/1120 validated...')

print(f'\nValidation complete:')
print(f'  Valid     : {len(valid_recs)}')
print(f'  Corrupted : {len(corrupted_recs)}')
print(f'  Modes seen: {modes_seen}')

if corrupted_recs:
    print('Corrupted files:')
    for r in corrupted_recs:
        print(f'  {r["image_path"]}  —  {r["preprocessing_status"]}')

---
## Section 11 — Duplicate Detection

In [ ]:
print('Computing partial MD5 hashes for duplicate detection...')

for rec in valid_recs:
    try:
        with open(rec['image_path'], 'rb') as f:
            rec['md5_partial'] = hashlib.md5(f.read(8192)).hexdigest()
    except Exception:
        rec['md5_partial'] = ''

hash_counts = Counter(r['md5_partial'] for r in valid_recs if r['md5_partial'])
dup_hashes  = {k: v for k, v in hash_counts.items() if v > 1}

print(f'Duplicate partial-MD5 hashes : {len(dup_hashes)}')
if dup_hashes:
    for h, c in list(dup_hashes.items())[:5]:
        dups = [str(r['image_path'])[-40:] for r in valid_recs if r['md5_partial'] == h]
        print(f'  {h[:12]}...  count={c}  files={dups[:2]}')
else:
    print('  No duplicate images detected.')

---
## Section 12 — Copy Images to mura\_controls/

In [ ]:
# Create destination directories
for split_name in ['train', 'val', 'test']:
    (DEST_ROOT / split_name).mkdir(parents=True, exist_ok=True)

print('Copying images...')
print('(Original MURA files are NOT modified — shutil.copy2 creates a new copy)')
print()

manifest_rows = []
copy_counts   = {'train': 0, 'val': 0, 'test': 0}

for rec in valid_recs:
    split_name = rec['split']
    src_path   = rec['image_path']
    pid        = rec['patient_id']
    study_id   = rec['study_id']

    # Destination filename: MURA_{patient_id}_{study_id}_{original_name}
    dest_fname = f"MURA_{pid}_{study_id}_{src_path.name}"
    dest_path  = DEST_ROOT / split_name / dest_fname

    # Copy only if destination does not already exist (idempotent)
    if not dest_path.exists():
        shutil.copy2(src_path, dest_path)
    copy_counts[split_name] += 1

    manifest_rows.append({
        'filename'           : dest_fname,
        'image_path'         : str(dest_path),
        'split'              : split_name,
        'label'              : 0,
        'label_meaning'      : 'MURA-negative (normal/control) — NOT clinically confirmed Non-RA',
        'source_dataset'     : 'MURA',
        'body_region'        : 'XR_HAND',
        'patient_id'         : pid,
        'study_id'           : study_id,
        'original_mura_path' : str(src_path),
        'img_mode'           : rec.get('img_mode', ''),
        'img_size'           : rec.get('img_size', ''),
        'md5_partial'        : rec.get('md5_partial', ''),
        'preprocessing_status': rec['preprocessing_status'],
    })

for sp, ct in copy_counts.items():
    print(f'  {sp:<6}: {ct} images copied → {DEST_ROOT / sp}')

print()
print('Copy complete. Original MURA files untouched.')

---
## Section 13 — Save Manifest CSV

In [ ]:
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_PATH, index=False)

print(f'Manifest saved → {MANIFEST_PATH}')
print(f'  Rows    : {len(manifest_df)}')
print(f'  Columns : {list(manifest_df.columns)}')
print()
print('Sample rows:')
print(manifest_df[['filename','split','label','patient_id','study_id',
                    'body_region','img_mode','preprocessing_status']].head(5).to_string(index=False))

---
## Section 14 — Preprocessing Compatibility Check

Verify that MURA images can be processed through the existing RA preprocessing pipeline
without any code modifications.

In [ ]:
# Load existing RA normalisation stats
norm_stats = pd.read_csv(REPORTS_DIR / 'normalisation_stats.csv')
mean_vals  = norm_stats['mean'].tolist()
std_vals   = norm_stats['std'].tolist()

print('RA normalisation stats (from Notebook 04):')
print(norm_stats.to_string(index=False))
print()

# Test the existing transform pipeline on one MURA image
import sys
sys.path.insert(0, str(PROJECT_ROOT))
import torch
from src.dataset import get_transforms

test_transform = get_transforms('test', (224, 224), mean_vals, std_vals)

sample_mura_path = manifest_df[manifest_df['split']=='train']['image_path'].iloc[0]
with Image.open(sample_mura_path) as img:
    img_rgb = img.convert('RGB')   # same as RADataset.__getitem__
    tensor  = test_transform(img_rgb)

print(f'Sample MURA image processed through existing RA transform pipeline:')
print(f'  Source path  : ...{sample_mura_path[-50:]}')
print(f'  Output shape : {tensor.shape}   (expected [3, 224, 224])')
print(f'  dtype        : {tensor.dtype}')
print(f'  Value range  : [{tensor.min():.4f}, {tensor.max():.4f}]')
print()
assert tensor.shape == torch.Size([3, 224, 224]), 'Unexpected tensor shape'
print('PASS: MURA images are fully compatible with the existing RA preprocessing pipeline.')
print('      No changes to src/ or existing notebooks required.')

---
## Section 15 — Sample Image Visualisation

In [ ]:
# Display 2 RA images vs 2 MURA control images side by side
ra_samples   = ra_train[ra_train['isRA']==1].sample(2, random_state=42)
mura_samples = manifest_df[manifest_df['split']=='train'].sample(2, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('RA Images (left) vs MURA XR_HAND Normal Controls (right)',
             fontsize=13, fontweight='bold')

for i, (_, row) in enumerate(ra_samples.iterrows()):
    with Image.open(row['full_path']) as img:
        thumb = img.convert('RGB')
        thumb.thumbnail((400, 400))
    axes[i][0].imshow(thumb, cmap='gray')
    axes[i][0].set_title(f'RA image (label=1)\n{row["filename"][:40]}', fontsize=8)
    axes[i][0].axis('off')

for i, (_, row) in enumerate(mura_samples.iterrows()):
    with Image.open(row['image_path']) as img:
        thumb = img.copy()
        thumb.thumbnail((400, 400))
    axes[i][1].imshow(thumb)
    axes[i][1].set_title(
        f'MURA XR_HAND Normal (label=0)\n'
        f'{row["patient_id"]} / {row["study_id"]}\n'
        f'mode={row["img_mode"]}  size={row["img_size"]}', fontsize=8)
    axes[i][1].axis('off')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_ra_vs_mura_samples.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: 07_ra_vs_mura_samples.png')

---
## Section 16 — Final Verification Table

In [ ]:
# ── Reload manifest from disk (verify the saved file is correct) ──────────────
manifest_check = pd.read_csv(MANIFEST_PATH)

ra_counts_by_split = {
    'train': int((ra_train['isRA']==1).sum()),
    'val'  : int((ra_val  ['isRA']==1).sum()),
    'test' : int((ra_test ['isRA']==1).sum()),
}

print('=' * 70)
print('FINAL VERIFICATION TABLE')
print('=' * 70)
print(f'  {"Split":<8}  {"RA":>6}  {"MURA_ctrl":>10}  {"Total":>7}  '
      f'{"Ctrl_pts":>9}  {"Balanced?"}')
print('-' * 65)

all_ok   = True
grand_ra = grand_ctrl = grand_tot = 0

for sp in ['train', 'val', 'test']:
    ra_n    = ra_counts_by_split[sp]
    ctrl_df = manifest_check[(manifest_check['split']==sp) &
                              (manifest_check['preprocessing_status']=='ok')]
    ctrl_n  = len(ctrl_df)
    ctrl_pts= ctrl_df['patient_id'].nunique()
    total   = ra_n + ctrl_n
    balanced= ra_n == ctrl_n

    if not balanced or ctrl_n != TARGETS[sp]:
        all_ok = False

    grand_ra   += ra_n
    grand_ctrl += ctrl_n
    grand_tot  += total

    print(f'  {sp:<8}  {ra_n:>6}  {ctrl_n:>10}  {total:>7}  '
          f'{ctrl_pts:>9}  {"YES" if balanced else "NO"}')

print('-' * 65)
print(f'  {"TOTAL":<8}  {grand_ra:>6}  {grand_ctrl:>10}  {grand_tot:>7}  '
      f'{manifest_check["patient_id"].nunique():>9}')
print()
print(f'Expected totals — Train:1438  Val:272  Test:530  Grand:2240')
print(f'Actual totals   — Train:{ra_counts_by_split["train"]+TARGETS["train"]}  '
      f'Val:{ra_counts_by_split["val"]+TARGETS["val"]}  '
      f'Test:{ra_counts_by_split["test"]+TARGETS["test"]}  '
      f'Grand:{grand_tot}')
print()

# Additional checks
val_p   = set(manifest_check[manifest_check['split']=='val']['patient_id'])
test_p  = set(manifest_check[manifest_check['split']=='test']['patient_id'])
train_p = set(manifest_check[manifest_check['split']=='train']['patient_id'])

leak_vt  = val_p   & test_p
leak_vtr = val_p   & train_p
leak_ttr = test_p  & train_p

all_imgs = manifest_check['filename'].tolist()
img_dups = len(all_imgs) - len(set(all_imgs))

print(f'Patient leakage (val∩test)  : {len(leak_vt)}')
print(f'Patient leakage (val∩train) : {len(leak_vtr)}')
print(f'Patient leakage (test∩train): {len(leak_ttr)}')
print(f'Image duplicates            : {img_dups}')
print(f'Corrupted images            : {(manifest_check["preprocessing_status"]!="ok").sum()}')
print(f'Body regions used           : {manifest_check["body_region"].unique().tolist()}')
print(f'Source dataset              : {manifest_check["source_dataset"].unique().tolist()}')
print(f'Label value(s)              : {manifest_check["label"].unique().tolist()}')
print()

leakage_ok = (len(leak_vt)==0 and len(leak_vtr)==0 and len(leak_ttr)==0)

if all_ok and leakage_ok and img_dups == 0 and grand_tot == 2240:
    print('=' * 70)
    print('FINAL VERIFICATION: PASS')
    print('Dataset is ready. No training code modified.')
    print('=' * 70)
else:
    issues = []
    if not all_ok:      issues.append('count mismatch')
    if not leakage_ok:  issues.append('patient leakage')
    if img_dups > 0:    issues.append('duplicate images')
    if grand_tot != 2240: issues.append(f'grand total {grand_tot} != 2240')
    print('FINAL VERIFICATION: FAIL —', ', '.join(issues))

In [ ]:
# ── Summary visualisation ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Combined Dataset — RA vs MURA Control per Split',
             fontsize=13, fontweight='bold')

palette = {'RA (label=1)': '#4C72B0', 'MURA Control (label=0)': '#DD8452'}

for ax, sp in zip(axes, ['train', 'val', 'test']):
    ra_n   = ra_counts_by_split[sp]
    ctrl_n = len(manifest_check[(manifest_check['split']==sp) &
                                 (manifest_check['preprocessing_status']=='ok')])
    labels  = ['RA (label=1)', 'MURA Control\n(label=0)']
    counts  = [ra_n, ctrl_n]
    colours = ['#4C72B0', '#DD8452']
    bars    = ax.bar(labels, counts, color=colours, edgecolor='white', width=0.5)
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 2,
                str(int(b.get_height())), ha='center', va='bottom', fontsize=11)
    ax.set_title(f'{sp.capitalize()}  (n={ra_n+ctrl_n})')
    ax.set_ylabel('Images')
    ax.set_ylim(0, max(counts) * 1.2)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_combined_dataset_splits.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: 07_combined_dataset_splits.png')

---
## Section 17 — Output File Checklist

In [ ]:
print('=== OUTPUT FILES ===')

check_files = [
    MANIFEST_PATH,
    DEST_ROOT / 'train',
    DEST_ROOT / 'val',
    DEST_ROOT / 'test',
    PROJECT_ROOT / 'outputs' / 'plots' / '07_mura_pos_neg_split.png',
    PROJECT_ROOT / 'outputs' / 'plots' / '07_ra_vs_mura_samples.png',
    PROJECT_ROOT / 'outputs' / 'plots' / '07_combined_dataset_splits.png',
]

for p in check_files:
    p = Path(p)
    if p.is_dir():
        n = len(list(p.iterdir()))
        print(f'  [OK  DIR ] {str(p).replace(str(PROJECT_ROOT), "")}  ({n} files)')
    elif p.exists():
        kb = round(p.stat().st_size / 1024, 1)
        print(f'  [OK  FILE] {str(p).replace(str(PROJECT_ROOT), "")}  ({kb} KB)')
    else:
        print(f'  [MISSING ] {str(p).replace(str(PROJECT_ROOT), "")}')

print()
print('=== FILES NOT MODIFIED (verified unchanged) ===')
unchanged = [
    RA_TRAIN_CSV, RA_VAL_CSV, RA_TEST_CSV,
    PROJECT_ROOT / 'notebooks' / '05_train_ra_cnn.ipynb',
    PROJECT_ROOT / 'src' / 'dataset.py',
    PROJECT_ROOT / 'src' / 'model.py',
]
for p in unchanged:
    print(f'  [UNTOUCHED] {Path(p).name}')

---
## Section 18 — Limitations and Next Steps

### Limitations

1. **MURA ≠ Non-RA confirmation.** MURA-negative images were labelled *normal* for musculoskeletal abnormalities in general. They are NOT clinically confirmed to be free of Rheumatoid Arthritis. They are used as *radiographically normal hand X-ray controls*.

2. **Different acquisition protocols.** MURA images come from Stanford Medicine (USA); RA images come from Japanese hospitals. Scanner models, exposure settings, and radiograph positioning may differ systematically. This is a known domain-shift limitation.

3. **Image mode difference.** RA images are grayscale BMP (`mode=L`); MURA images are RGB PNG. The existing `RADataset` converts all images to RGB via `.convert('RGB')`, which handles this transparently — but it is worth noting.

4. **No age/sex/clinical metadata for MURA.** The MURA dataset does not provide patient demographic information. The multimodal metadata columns (`age_norm`, `Sex_enc`, etc.) will be 0/missing for MURA controls in any combined experiment.

5. **Test set balance.** The test set is now 265 RA vs 265 MURA controls — perfectly balanced. However, a model trained on this combined dataset should be evaluated with awareness that the "non-RA" class is MURA-normal, not clinically-confirmed non-RA.

### Next steps (not done here)

- Create a new training notebook (e.g. `08_train_ra_mura_combined.ipynb`) that loads both the RA manifest and the MURA controls manifest together.
- The existing `src/dataset.py` and `src/model.py` do not need modification — the `SubsetRADataset` from Notebook 05B can read the combined manifests directly.
- Re-run normalisation statistics on the combined training set (MURA RGB images may shift the channel mean/std).
- Consider domain adaptation or separate batch normalisation for the two data sources.